# full_3 — Channel-flow (A1) training via `train_channel.py`

**Default: scalar-Q smoke test.** Run all cells to train for two epochs on five snapshots per split, using a coarsened XZ grid. This uses the local A1 S3 dataset, writes checkpoints under a separate smoke-test project, and checks the saved Trackio metrics and images.

Set `SMOKE_TEST = False` in Config to restore the full velocity-run settings. Data is loaded once and can be reused for architecture trials.

In [1]:
import sys
from pathlib import Path

# Support kernels launched from either the notebook directory or the repo root.
cwd = Path.cwd()
for source_dir in (cwd, cwd / "networks" / "full"):
    if (source_dir / "train_channel.py").is_file():
        if str(source_dir) not in sys.path:
            sys.path.insert(0, str(source_dir))
        break
else:
    raise FileNotFoundError("Start the kernel from the repo root or networks/full.")

from train_channel import (
    load_training_data,
    run_trial,
    EPOCHS,
    BATCH_SIZE,
    NUM_WORKERS,
    S3_MODEL_PREFIX,
    S3_BUCKET,
    STORE_PATH,
)
print("train_channel imported OK")
print(f"  default EPOCHS   = {EPOCHS}")
print(f"  BATCH_SIZE       = {BATCH_SIZE}")
print(f"  NUM_WORKERS      = {NUM_WORKERS}")
print(f"  S3 dataset       = s3://{S3_BUCKET}/{STORE_PATH}")
print(f"  S3 model prefix  = {S3_MODEL_PREFIX}")

train_channel imported OK
  default EPOCHS   = 20
  BATCH_SIZE       = 4
  NUM_WORKERS      = 16
  S3 dataset       = s3://simulations/flow_export/A1
  S3 model prefix  = models/channel-1


## Config

In [2]:
SMOKE_TEST = True
LR_INIT = 1e-3

if SMOKE_TEST:
    EPOCHS_RUN = 2
    MAX_SAMPLES = 5  # One full batch and one short batch per split.
    TASK = "wall_pressure_to_q"
    XZ_STRIDE = 4
    ARCH = {"c": 8, "n_blocks": 1}
    PROJECT = "channel_full3_q_smoke"
    LABEL = "scalar-q-smoke"
    SPECTRAL_IMAGE_EVERY = 1
else:
    EPOCHS_RUN = 20
    MAX_SAMPLES = 5000
    TASK = "wall_pressure_to_velocity"
    XZ_STRIDE = 1
    ARCH = {"c": 64, "n_blocks": 3}
    PROJECT = "channel_full3_ptoq"
    LABEL = "full3"
    SPECTRAL_IMAGE_EVERY = 5

WALL = None  # Defaults to the lower wall for wall-pressure tasks.
HALF_Y = True
SPECTRAL_IMAGE_SAMPLES = 2

print(f"{TASK}: epochs={EPOCHS_RUN}, max_samples={MAX_SAMPLES}, xz_stride={XZ_STRIDE}, arch={ARCH}")

wall_pressure_to_q: epochs=2, max_samples=5, xz_stride=4, arch={'c': 8, 'n_blocks': 1}


## Load data once

Rerun the load cell when changing the task, wall, sample limit, or spatial grid, or after restarting the kernel.

In [3]:
import os; print(os.getpid())

2690392


In [3]:
training_data = load_training_data(
    max_samples=MAX_SAMPLES, task=TASK, wall=WALL,
    half_y=HALF_Y, xz_stride=XZ_STRIDE, log=True,
)
print(f"Validation target: {tuple(training_data.val[1].shape)} (T, C, X, Y, Z)")

  task=wall_pressure_to_q  wall=lower  half_y=True  xz_stride=4  T=4001  max_samples=5
  train: [60:65:1] (5/3136 snapshots; loading)
  val: [3205:3210:1] (5/392 snapshots; loading)
  test: [3607:3999:1] (392/392 snapshots; held out, not loaded)
  [load] 5 snapshots  x=0.00 GiB  y=0.00 GiB  peak RSS=0.79 GiB
  [load] 5 snapshots  x=0.00 GiB  y=0.00 GiB  peak RSS=0.79 GiB
Validation target: (5, 1, 32, 65, 32) (T, C, X, Y, Z)


Load Zarr:   0%|          | 0/1 [00:00<?, ?chunk/s]

Load Zarr:   0%|          | 0/1 [00:00<?, ?chunk/s]

## Run the trial

Change `ARCH` in the config cell, then rerun this cell without reloading the data.

In [4]:
result = run_trial(
    ARCH,
    data=training_data,
    epochs=EPOCHS_RUN,
    lr=LR_INIT,
    trackio_project=PROJECT,
    trial_label=LABEL,
    spectral_image_samples=SPECTRAL_IMAGE_SAMPLES,
    spectral_image_every=SPECTRAL_IMAGE_EVERY,
    log=True,
)

[trial] {'arch': {'c': 8, 'n_blocks': 1, 'dy': 65, 'out_ch': 1}, 'task': 'wall_pressure_to_q', 'wall': 'lower', 'half_y': True, 'xz_stride': 4, 'epochs': 2, 'lr': 0.001}
* Trackio project initialized: channel_full3_q_smoke
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* View dashboard by running in your terminal:
trackio show --project "channel_full3_q_smoke"
* or by running in Python: trackio.show(project="channel_full3_q_smoke")
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: dainty-sunset-0
  model: 0.00M params
  epoch   0  train=1.2643  val=30.5555  lr=1.00e-03
  epoch   1  train=1.4141  val=30.5485  lr=1.00e-03
* Run finished. Uploading logs to Trackio (please wait...)


Train:   0%|          | 0/2 [00:00<?, ?batch/s]

Validation:   0%|          | 0/2 [00:00<?, ?batch/s]

Train:   0%|          | 0/2 [00:00<?, ?batch/s]

Validation:   0%|          | 0/2 [00:00<?, ?batch/s]

## Result

In [5]:
import json

# strip the live model out of the printable summary
summary = {k: v for k, v in result.items() if k != "model"}
print(json.dumps(summary, indent=2))
print()
print(f"best val MSE : {result['val_mse']:.6f}")
print(f"best epoch   : {result['best_epoch']}")
print(f"n params     : {result['n_params']:,}")
print(f"checkpoint   : {result['checkpoint_uri']}")

{
  "val_mse": 30.54853057861328,
  "best_epoch": 1,
  "n_params": 3913,
  "arch": {
    "c": 8,
    "n_blocks": 1,
    "dy": 65,
    "out_ch": 1
  },
  "task": "wall_pressure_to_q",
  "wall": "lower",
  "half_y": true,
  "xz_stride": 4,
  "epochs": 2,
  "lr": 0.001,
  "s3_prefix": "models/channel-1/channel_full3_q_smoke/dainty-sunset-0",
  "s3_best_tag": "best",
  "checkpoint_uri": "s3://models/channel-1/channel_full3_q_smoke/dainty-sunset-0/ckpt_best.pt",
  "interrupted_checkpoint_uri": null
}

best val MSE : 30.548531
best epoch   : 1
n params     : 3,913
checkpoint   : s3://models/channel-1/channel_full3_q_smoke/dainty-sunset-0/ckpt_best.pt


In [6]:
if SMOKE_TEST:
    import math
    import trackio

    assert result["arch"]["out_ch"] == training_data.val[1].shape[1] == 1
    assert math.isfinite(result["val_mse"])
    assert result["checkpoint_uri"] is not None

    # Verify the persisted Trackio payload for this run, not just the model output.
    run_name = result["s3_prefix"].rsplit("/", 1)[-1]
    rows = trackio.SQLiteStorage.get_logs(PROJECT, run_name)
    epochs = {row["epoch"]: row for row in rows if "epoch" in row and "val/mse" in row}
    assert set(epochs) == set(range(EPOCHS_RUN)), "Missing Trackio validation logs"
    for row in epochs.values():
        bands = [value for key, value in row.items() if key.startswith("val/spectral_mse/")]
        assert bands and math.isclose(sum(bands), row["val/mse"], rel_tol=1e-5, abs_tol=1e-7)

    final_row = epochs[EPOCHS_RUN - 1]
    image_paths = sorted(trackio.get_project_media_path(PROJECT, run_name).rglob("*.png"))
    expected_images = min(SPECTRAL_IMAGE_SAMPLES, len(training_data.val[1]))
    assert sum(key.startswith("val/spectral_xz/") for key in final_row) == expected_images
    assert len(image_paths) == EPOCHS_RUN * expected_images
    print(f"Smoke test passed: scalar Q, {len(epochs)} spectral metric steps, {len(image_paths)} saved images.")
    print(f'Dashboard: trackio show --project "{PROJECT}"')

Smoke test passed: scalar Q, 2 spectral metric steps, 4 saved images.
Dashboard: trackio show --project "channel_full3_q_smoke"
